# TP : Reproduire UniMER sur MELD (version texte seul)

**Article de référence** : *UniMER: Multilingual Multimodal Emotion Recognition with Prototype-Guided Attention Pooling* (EMNLP 2026).

## Objectifs

- Comprendre la reconnaissance des émotions dans une conversation (ERC).
- Préparer des données conversationnelles avec un contexte (les k énoncés précédents).
- Adapter un grand modèle de langage (Gemma-3-12B) avec QLoRA, sur un GPU Colab gratuit.
- Entraîner une tête de classification avec une perte pondérée pour gérer le déséquilibre des classes.
- Évaluer avec les métriques du papier : accuracy, F1 pondéré et F1 macro.

## Prérequis

- Python et PyTorch de base.
- Un compte Hugging Face, un token d'accès, et l'acceptation de la licence Gemma sur la page du modèle `google/gemma-3-12b-it`.
- Un runtime Colab avec GPU : *Exécution > Modifier le type d'exécution > GPU*.

## Ce qui est simplifié par rapport au papier

| Élément | Papier | Ce TP | Raison |
|---|---|---|---|
| Matériel | A100 80 Go | T4 16 Go | Contrainte Colab gratuit |
| Précision du modèle | bfloat16 | 4 bits NF4 (QLoRA) | Tenir en mémoire |
| Modalités | texte + captions vidéo | texte seul | Pas de prétraitement vidéo |
| Longueur de contexte | 40 énoncés | 8 énoncés | Temps de calcul |
| Prototypes par classe | oui | non (partie bonus) | Simplicité pour la première version |
| Graines | 5 | 1 | Temps de calcul |

## Livrables

1. Ce notebook exécuté, avec toutes les sorties visibles.
2. Un tableau de résultats (accuracy, F1 pondéré, F1 macro, F1 par classe) sur le jeu de test.
3. Une réponse écrite (une demi-page) aux questions de la dernière section.


## Étape 1 : vérifier le GPU et installer les paquets

`nvidia-smi` affiche la carte graphique attribuée par Colab. Une T4 (16 Go) suffit pour ce TP, mais il faut savoir qu'elle ne gère pas bfloat16 : on utilise donc float16.

Les versions sont fixées pour éviter les incompatibilités entre `transformers` et `peft`.

In [ ]:
!nvidia-smi
!pip install -q transformers==4.51.3 peft==0.14.0 accelerate bitsandbytes datasets scikit-learn pandas


## Étape 2 : récupérer et explorer MELD

MELD contient des extraits de la série *Friends*. Chaque ligne est un énoncé avec un locuteur, une émotion et un identifiant de dialogue.

Observez le **déséquilibre des classes** : *neutral* domine, alors que *fear* et *disgust* sont très rares. C'est le problème central du papier. Un modèle qui prédit toujours *neutral* aurait déjà une accuracy élevée, ce qui justifie d'utiliser aussi le F1 macro.

In [ ]:
!git clone -q https://github.com/declare-lab/MELD.git

import pandas as pd

D = 'MELD/data/MELD/'
train = pd.read_csv(D + 'train_sent_emo.csv')
dev = pd.read_csv(D + 'dev_sent_emo.csv')
test = pd.read_csv(D + 'test_sent_emo.csv')

LABELS = ['neutral', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
l2i = {l: i for i, l in enumerate(LABELS)}

print('Tailles (train, dev, test) :', train.shape[0], dev.shape[0], test.shape[0])
print(train['Emotion'].value_counts())
train.head(3)


## Étape 3 : construire l'entrée avec le contexte

Une émotion se comprend rarement sur une seule phrase. Pour chaque énoncé, on ajoute les **k énoncés précédents** du même dialogue. Le papier suit la même logique (`[CTX] ... [UTT] ...`), avec k = 40 sur MELD. Ici on prend k = 8 pour que l'entraînement reste rapide.

On trie d'abord par `Utterance_ID` pour respecter l'ordre chronologique.

In [ ]:
def build(df, k=8):
    rows = []
    for _, g in df.groupby('Dialogue_ID', sort=False):
        g = g.sort_values('Utterance_ID')
        ut = g['Utterance'].tolist()
        sp = g['Speaker'].tolist()
        em = g['Emotion'].tolist()
        for i in range(len(ut)):
            ctx = ' '.join(f'{sp[j]}: {ut[j]}' for j in range(max(0, i - k), i))
            text = f'Contexte: {ctx}\nÉnoncé: {sp[i]}: {ut[i]}'
            rows.append({'text': text, 'label': l2i[em[i]]})
    return pd.DataFrame(rows)

train_df = build(train, k=8)
dev_df = build(dev, k=8)
test_df = build(test, k=8)

print(train_df['text'].iloc[50])
print('Label :', LABELS[train_df['label'].iloc[50]])


## Étape 4 : charger Gemma-3-12B en 4 bits avec LoRA

- **QLoRA** : le modèle de base est quantifié en 4 bits et gelé. Seuls les petits adaptateurs LoRA sont entraînés. C'est ce qui rend le modèle utilisable sur une T4.
- **LoRA** écrit la mise à jour de chaque matrice comme $W = W_0 + BA$ avec un rang r = 64. Seules $A$ et $B$ sont entraînées.
- On utilise `AutoModel` et non `AutoModelForCausalLM` : nous n'avons pas besoin de prédire le prochain mot, donc inutile de calculer les logits sur les 262 000 tokens du vocabulaire.
- Le `target_modules` est une expression régulière qui cible uniquement la partie langage, pas le encodeur d'images.

Le chargement prend plusieurs minutes. Si vous obtenez une erreur d'authentification, vérifiez votre token Hugging Face et la licence Gemma.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

login()  # collez votre token Hugging Face

MODEL_ID = 'google/gemma-3-12b-it'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,  # T4 : float16 (pas bfloat16)
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModel.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')

lora = LoraConfig(
    r=64, lora_alpha=128, lora_dropout=0.05, bias='none',
    target_modules=r'.*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)',
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()  # comparez avec les 2,4 % du papier


## Étape 5 : la tête de classification et la perte pondérée

**Représentation** : on fait la moyenne des états cachés de chaque token (mean pooling, en ignorant le padding). On obtient un vecteur par énoncé, qui passe dans une couche linéaire à 7 sorties.

**Pondération des classes** : le papier utilise les poids « nombre effectif d'échantillons » (Cui et al., 2019) :

$$w_k = \frac{1 - \beta}{1 - \beta^{n_k}}, \quad \beta = 0{,}9999$$

Une classe rare reçoit un poids plus grand, donc une erreur sur *fear* coûte plus cher qu'une erreur sur *neutral*. On ajoute aussi un lissage des labels (ε = 0,15) pour limiter la sur-confiance.

In [ ]:
from torch import nn

hidden = base.config.text_config.hidden_size
head = nn.Linear(hidden, len(LABELS)).to(model.device)

counts = train['Emotion'].map(l2i).value_counts().sort_index().values
beta = 0.9999
eff = (1 - beta) / (1 - beta ** counts)
w = eff / eff.sum() * len(LABELS)
print('Poids par classe :', dict(zip(LABELS, w.round(2))))

loss_fn = nn.CrossEntropyLoss(
    weight=torch.tensor(w, dtype=torch.float32).to(model.device),
    label_smoothing=0.15,
)

def encode(texts):
    enc = tok(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
    h = model(**enc).last_hidden_state.float()
    m = enc['attention_mask'].unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)  # moyenne sans le padding


## Étape 6 : entraîner

On utilise un **batch effectif de 48** : un micro-batch de 4, accumulé sur 12 pas. Cela reproduit la taille de batch du papier avec la mémoire d'une T4.

Le papier entraîne 1 époque sur MELD avec un taux d'apprentissage de 5e-5 et une décroissance cosinus. Ici, on garde la même valeur de taux, sans scheduler, pour simplifier.

Si vous manquez de mémoire, décommentez `gradient_checkpointing_enable()`. C'est plus lent, mais cela réduit beaucoup la mémoire.

In [ ]:
# model.gradient_checkpointing_enable()  # décommenter si erreur 'CUDA out of memory'

BATCH, ACC = 4, 12
params = [p for p in model.parameters() if p.requires_grad] + list(head.parameters())
opt = torch.optim.AdamW(params, lr=5e-5, weight_decay=0.01)

model.train()
idx = torch.randperm(len(train_df)).tolist()
for step, s in enumerate(range(0, len(idx), BATCH)):
    b = train_df.iloc[idx[s:s + BATCH]]
    logits = head(encode(b['text'].tolist()))
    y = torch.tensor(b['label'].values, device=model.device)
    loss = loss_fn(logits, y) / ACC
    loss.backward()
    if (step + 1) % ACC == 0:
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        opt.zero_grad()
    if step % 200 == 0:
        print(f'pas {step} / {len(idx) // BATCH} | perte {loss.item() * ACC:.3f}')

opt.step()
opt.zero_grad()


## Étape 7 : évaluer

Trois métriques, à interpréter ensemble :
- **Accuracy** : sensible à la classe majoritaire.
- **F1 pondéré (wF1)** : moyenne des F1 pondérée par la taille des classes. C'est la métrique principale du papier sur MELD.
- **F1 macro (mF1)** : moyenne simple des F1 par classe. Elle révèle l'échec sur les classes rares.

Comparez vos chiffres à ceux du papier : **accuracy 71,1 %, wF1 70,7 %, mF1 58,7 %**. Attention, ces chiffres incluent la vidéo, donc un écart est attendu.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

model.eval()
preds = []
with torch.no_grad():
    for s in range(0, len(test_df), BATCH * 2):
        b = test_df.iloc[s:s + BATCH * 2]
        preds += head(encode(b['text'].tolist())).argmax(-1).cpu().tolist()

y_true = test_df['label'].values
print(f'Accuracy : {accuracy_score(y_true, preds):.3f}')
print(f'F1 pondéré : {f1_score(y_true, preds, average="weighted"):.3f}')
print(f'F1 macro : {f1_score(y_true, preds, average="macro"):.3f}')
print(classification_report(y_true, preds, target_names=LABELS, zero_division=0))


## Étape 8 : sauvegarder

Les sessions Colab se coupent. Sauvegardez les adaptateurs LoRA et la tête dans Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/unimer_meld'
model.save_pretrained(OUT + '/lora')
torch.save(head.state_dict(), OUT + '/head.pt')
print('Sauvegardé dans', OUT)


## Questions et travail à rendre

**Partie A (obligatoire)**

1. Calculez la performance d'un classifieur qui prédit toujours *neutral* sur le jeu de test. Quelle accuracy obtenez-vous ? Pourquoi l'accuracy seule est-elle trompeuse ici ?
2. Présentez votre tableau de résultats et comparez-le à celui du papier. Quels écarts attribuez-vous à la absence de vidéo, à la réduction de k et à la quantification ?
3. Quelle classe a le F1 le plus faible ? Pourquoi ? Regardez les exemples mal classés de cette classe et expliquez l'erreur.
4. Expliquez pourquoi on utilise la moyenne de `last_hidden_state` plutôt que le dernier token.

**Partie B (au choix, une seule expérience à réaliser)**

5. **Effet du contexte** : refaites l'entraînement avec k = 0 puis k = 20. Tracez le F1 macro en fonction de k.
6. **Effet de la pondération** : refaites l'entraînement avec une perte non pondérée. Quelle classe souffre le plus ?
7. **Bonus, prototypes** : remplacez la moyenne par le *class-prototype attention pooling* du papier. Pour chaque classe k, calculez $e_t^{(k)} = p_k^\top \tanh(W_h h_t + b_h)$, puis un softmax sur les tokens, puis $c_k = \sum_t \alpha_t^{(k)} h_t$. Concaténez les $c_k$ et passez-les dans un MLP. Comparez le F1 macro à la version simple.

**Critères d'évaluation** : exécution complète du notebook (40 %), qualité de l'analyse des résultats (40 %), expérience de la partie B (20 %).
